## Import libs

In [32]:
import numpy as np
import pandas as pd
import random
import torch
import wandb
from transformers import AutoTokenizer
from datasets import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn

## Set Seed for expermients

In [33]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed(SEED)

## Setup and Config

In [34]:
WANDB_PROJECT = "21f2000763-t22026"
TOKENIZER = "google-bert/bert-base-uncased"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)

CURRENT_RUN_NAME = "bilstm_1_01_baseline"
CONFIG = {
    "embed_dim": 128,
    "hidden_dim": 128,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": 16,
    "epochs": 5
}

# CURRENT_RUN_NAME = "bilstm_02_high_dropout"
# CONFIG = {
#     "embed_dim": 128,
#     "hidden_dim": 128,
#     "dropout": 0.5,
#     "learning_rate": 1e-3,
#     "batch_size": 16,
#     "epochs": 5
# }

# CURRENT_RUN_NAME = "bilstm_03_increase_dim"
# CONFIG = {
#     "embed_dim": 256,      # Doubled
#     "hidden_dim": 256,     # Doubled
#     "dropout": 0.5,
#     "learning_rate": 1e-3,
#     "batch_size": 16,
#     "epochs": 5
# }

# CURRENT_RUN_NAME = "bilstm_04_lr"
# CONFIG = {
#     "embed_dim": 256,
#     "hidden_dim": 256,
#     "dropout": 0.5,
#     "learning_rate": 5e-4, # in half
#     "batch_size": 16,
#     "epochs": 5
# }



In [35]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

## Dataset creation

In [36]:
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df["labels"] = train_df["answer"].map(label_map)

train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

## Tokenize function

In [37]:
def tokenize(batch):
  texts = [
    f"{p} [SEP] A: {a} B: {b} C: {c} D: {d} E: {e}"
    for p, a, b, c, d, e in zip(
        batch["prompt"],
        batch["A"],
        batch["B"],
        batch["C"],
        batch["D"],
        batch["E"],
    )
  ]

  tokenized = tokenizer(texts, padding="max_length", truncation = True, max_length=256)

  if 'labels' in batch:
    tokenized['labels'] = batch["labels"]

  return tokenized

In [38]:
train_dataset = train_dataset.map(tokenize, batched = True)
test_dataset  = test_dataset.map(tokenize, batched = True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Dataloader

In [39]:
split_dataset = train_dataset.train_test_split(test_size = 0.2, seed = SEED)
train_loader  = DataLoader(split_dataset["train"], batch_size = CONFIG["batch_size"], shuffle = True)
val_loader    = DataLoader(split_dataset["test"], batch_size = CONFIG["batch_size"])
test_loader   = DataLoader(test_dataset, batch_size = CONFIG["batch_size"])

## Model

In [40]:
class BiLSTMAttention(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(
        vocab_size,
        CONFIG["embed_dim"],
        padding_idx = 0
    )

    self.lstm = nn.LSTM(
        CONFIG["embed_dim"],
        CONFIG["hidden_dim"],
        batch_first = True,
        bidirectional = True
    )

    self.attention = nn.Linear(CONFIG["hidden_dim"] * 2 ,1)
    self.dropout   = nn.Dropout(CONFIG["dropout"])
    self.fc        = nn.Linear(CONFIG["hidden_dim"] * 2 ,5)

  def forward(self, input_ids, attention_mask):
    x = self.embedding(input_ids)
    x, _ = self.lstm(x)

    scores = self.attention(x).squeeze(-1)

    scores = scores.masked_fill(attention_mask == 0, -1e9)

    weights = torch.softmax(scores, dim=1)

    # Apply weights to LSTM outputs
    context = torch.sum(
        x * weights.unsqueeze(-1),
        dim = 1
    )

    context = self.dropout(context)
    return self.fc(context)

## MAP 3 metric function

In [41]:
def map3_score(predictions, labels):
  score = 0.0
  for i in range(len(labels)):
    pred = predictions[i]
    label = labels[i]

    if label in pred:
      rank = pred.index(label)
      score += 1.0/ (rank + 1)

  return score / len(labels)

## Training and Validation

In [42]:
wandb.init(project=WANDB_PROJECT, name = CURRENT_RUN_NAME, config = CONFIG)
model = BiLSTMAttention(tokenizer.vocab_size).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr = CONFIG["learning_rate"])
criterion = nn.CrossEntropyLoss()

print(f"Training on {DEVICE}....")

for epoch in range(CONFIG["epochs"]):
  model.train()
  running_loss = 0
  for batch in train_loader:
    # Set to zero
    optimizer.zero_grad()

    # forward pass
    logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
    loss = criterion(logits, batch["labels"].to(DEVICE))

    # back prop
    loss.backward()

    # update
    optimizer.step()

    running_loss += loss.item()

  train_loss = running_loss/len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  all_labels = []
  all_top3 = []

  val_correct, val_total = 0, 0
  with torch.no_grad():
    for batch in val_loader:
      logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))

      loss = criterion(logits, batch["labels"].to(DEVICE))
      val_loss += loss.item()
      probs = torch.softmax(logits, dim=1)
      top3 = torch.argsort(
          probs,
          descending=True
      )[:, :3]
      preds = top3[:, 0]
      labels = batch["labels"].to(DEVICE)
      val_correct += (preds == labels).sum().item()
      val_total += labels.size(0)


      all_labels.extend(labels.cpu().tolist())
      all_top3.extend(top3.cpu().tolist())

  val_loss = val_loss / len(val_loader)
  val_map3 = map3_score(all_top3, all_labels)
  val_acc = val_correct/val_total
  print(
    f"Epoch {epoch+1} | "
    f"Train Loss: {train_loss:.4f} | "
    f"Val Loss: {val_loss:.4f} | "
    f"Val Acc: {val_acc:.4f} | "
    f"Val MAP@3: {val_map3:.4f}"
  )
  wandb.log({
      'epoch': epoch + 1,
      "train_loss": train_loss,
      "val_loss": val_loss,
      "val_accuracy": val_acc,
      "val_map3": val_map3,
      "learning_rate": optimizer.param_groups[0]["lr"]
  })

wandb.finish()

Training on cuda....
Epoch 1 | Train Loss: 1.3778 | Val Loss: 0.7867 | Val Acc: 0.8025 | Val MAP@3: 0.8708
Epoch 2 | Train Loss: 0.4136 | Val Loss: 0.3015 | Val Acc: 0.9300 | Val MAP@3: 0.9621
Epoch 3 | Train Loss: 0.1095 | Val Loss: 0.0513 | Val Acc: 1.0000 | Val MAP@3: 1.0000
Epoch 4 | Train Loss: 0.0147 | Val Loss: 0.0097 | Val Acc: 1.0000 | Val MAP@3: 1.0000
Epoch 5 | Train Loss: 0.0039 | Val Loss: 0.0033 | Val Acc: 1.0000 | Val MAP@3: 1.0000


epoch,▁▃▅▆█
learning_rate,▁▁▁▁▁
train_loss,█▃▂▁▁
val_accuracy,▁▆███
val_loss,█▄▁▁▁
val_map3,▁▆███
epoch,5
learning_rate,0.001
train_loss,0.00394
val_accuracy,1
val_loss,0.00334


## Inference

In [43]:
print("Running Inference ....")

model.eval()
predictions = []

index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

with torch.no_grad():
  for batch in test_loader:
    logits = model(batch["input_ids"].to(DEVICE), batch['attention_mask'].to(DEVICE))
    probs = torch.nn.functional.softmax(logits, dim = 1)
    # Top 3 preds
    top3 = torch.argsort(probs, dim = 1, descending = True)[:, :3].cpu().numpy()
    for row in top3:
      predictions.append(" ".join([index_to_letter[idx] for idx in row]))

Running Inference ....


## Submission CSV

In [44]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission_df.to_csv("submission.csv", index = False)
submission_df.head()

,id,prediction
0,1,A C E
1,2,B C E
2,3,B E D
3,4,E A B
4,5,C D A
